# RSA Exploration — perceptXbind MEG

Time-resolved representational similarity analysis on rule-phase epochs.

**Scientific question**: Which representational structure best describes MEG signals during the rule phase — stimulus identity (which shape is shown), abstract role (whether the shape is playing the A or B role), or their conjunction?

**Pipeline**:
1. Assign condition labels using the `stim_by_role` scheme
2. Sanity-check condition counts and model RDM collinearity
3. Compute time-resolved neural RDMs (correlation distance + crossnobis)
4. Compare to model RDMs via Spearman ρ and partial regression
5. Visualise model RDM structure and time courses

In [ ]:
import sys, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

sys.path.insert(0, str(Path('..').resolve()))
from toolkit import (
    assign_conditions, condition_counts, RULE_PHASES, PHASE_OFFSETS,
    build_model_rdm, check_model_rdm_collinearity, time_resolved_rsa,
    CVSplitter,
)

print('toolkit loaded')

## 1. Load data

In [ ]:
EPOCHS_PATH = Path('../ported_results/sub-001_ses-01_task-binding_epo.fif')
BEHAV_PATH  = Path('/mnt/storage/NEU502B/brain-binding/data/2026-04-10/sub-001_events.csv')

epochs = mne.read_epochs(str(EPOCHS_PATH), preload=True)
df     = pd.read_csv(BEHAV_PATH).sort_values('trial').reset_index(drop=True)

X_all  = epochs.get_data(picks='meg').astype(np.float32)  # (1080, n_ch, n_t)
sfreq  = epochs.info['sfreq']
times  = epochs.times

print(f'Full epoch array : {X_all.shape}   (total × channels × timepoints)')
print(f'Sampling rate    : {sfreq} Hz')
print(f'Epoch window     : {times[0]:.3f} – {times[-1]:.3f} s')

## 2. Condition assignment

In [ ]:
cond_result = assign_conditions(df, scheme='stim_by_role')

print('Condition names:', cond_result['condition_names'])
print()
print('Condition metadata:')
display(cond_result['condition_metadata'])

In [ ]:
# Per-phase trial counts per condition
counts = condition_counts(cond_result, phase_filter=RULE_PHASES)
print('Trial counts per condition per phase:')
print(counts.pivot(index='condition_name', columns='phase', values='count').fillna(0).astype(int))

In [ ]:
# Select rule1 epochs for the primary RSA
# (rule1 is cleanest: all A-role, no ambiguity about which stimulus is shown)
ep_phase  = cond_result['epoch_phase']
cond_ids  = cond_result['condition_id']
rule1_mask = ep_phase == 'rule1'

X_rule1  = X_all[rule1_mask]           # (120, n_ch, n_t)
cids_r1  = cond_ids[rule1_mask]         # (120,)

print(f'rule1 epochs : {X_rule1.shape}')
print(f'Unique condition ids in rule1: {np.unique(cids_r1)}')
print('All are A-role:', all(
    cond_result['condition_metadata'].set_index('cond_id').loc[c, 'role'] == 'A'
    for c in np.unique(cids_r1)
))

## 3. Model RDMs

In [ ]:
meta = cond_result['condition_metadata']

# For rule1 epochs, only A-role conditions appear
# Filter metadata to those conditions
r1_cond_ids = np.unique(cids_r1)
meta_r1 = meta[meta['cond_id'].isin(r1_cond_ids)].reset_index(drop=True)
# Remap cond_ids to dense 0..K-1
remap = {old: new for new, old in enumerate(sorted(r1_cond_ids))}
cids_r1_dense = np.array([remap[c] for c in cids_r1])
meta_r1['cond_id'] = meta_r1['cond_id'].map(remap)

K = len(meta_r1)
print(f'K = {K} conditions in rule1')
print(meta_r1)

In [ ]:
# Build model RDMs (stimulus_identity and abstract_role are well-defined for rule1)
model_rdms = {
    'stimulus_identity':   build_model_rdm(meta_r1, 'stimulus_identity'),
    'conjunctive_binding': build_model_rdm(meta_r1, 'conjunctive_binding'),
}

# abstract_role has all-same role in rule1 (all A) → constant → skip for rule1
# We'll use all rule phases below where both roles appear

# VIF check
vif_df = check_model_rdm_collinearity(model_rdms)
print('VIF check:')
print(vif_df)

In [ ]:
# Visualise model RDMs
cond_labels = [f"{row['shape']}" for _, row in meta_r1.iterrows()]

fig, axes = plt.subplots(1, len(model_rdms), figsize=(4 * len(model_rdms), 4))
if len(model_rdms) == 1:
    axes = [axes]

for ax, (name, rdm) in zip(axes, model_rdms.items()):
    im = ax.imshow(rdm, cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_xticklabels(cond_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(K)); ax.set_yticklabels(cond_labels, fontsize=8)
    ax.set_title(name, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Model RDMs — rule1 conditions (A-role only)', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Full rule-phase RSA (all 3 positions, both roles)

Combining rule1+rule2+rule3 lets both `stimulus_identity` and `abstract_role` models have non-trivial structure.

In [ ]:
rule_mask = np.isin(ep_phase, RULE_PHASES)   # 360 epochs
X_rule    = X_all[rule_mask]
cids_rule = cond_ids[rule_mask]

# Filter out -1 (shouldn't be any for rule phases in stim_by_role)
valid = cids_rule >= 0
X_rule    = X_rule[valid]
cids_rule = cids_rule[valid]

# Dense-remap
classes_all = np.unique(cids_rule)
remap_all   = {old: new for new, old in enumerate(sorted(classes_all))}
cids_dense  = np.array([remap_all[c] for c in cids_rule])
meta_all    = meta[meta['cond_id'].isin(classes_all)].reset_index(drop=True)
meta_all['cond_id'] = meta_all['cond_id'].map(remap_all)
K_all = len(meta_all)

print(f'Rule-phase data : {X_rule.shape}')
print(f'K = {K_all} conditions (shapes × roles)')
print(meta_all.to_string())

In [ ]:
# Build model RDMs for the full stim_by_role scheme
model_rdms_all = {
    'stimulus_identity':   build_model_rdm(meta_all, 'stimulus_identity'),
    'abstract_role':       build_model_rdm(meta_all, 'abstract_role'),
    'conjunctive_binding': build_model_rdm(meta_all, 'conjunctive_binding'),
}

vif_all = check_model_rdm_collinearity(model_rdms_all)
print('VIF (all rule-phase models):')
print(vif_all)

# Visualise
cond_labels_all = [f"{row['shape']}_{row['role']}" for _, row in meta_all.iterrows()]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, rdm) in zip(axes, model_rdms_all.items()):
    im = ax.imshow(rdm, cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_xticks(range(K_all))
    ax.set_xticklabels(cond_labels_all, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(K_all))
    ax.set_yticklabels(cond_labels_all, fontsize=7)
    ax.set_title(name, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Model RDMs — all rule-phase conditions (shapes × roles)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Time-resolved RSA — Spearman ρ

In [ ]:
cv = CVSplitter(n_splits=5, stratified=True, shuffle=True, random_state=42)

print('Running Spearman RSA (correlation distance)...')
result_sp_corr = time_resolved_rsa(
    X_rule, cids_dense, model_rdms_all, times=times,
    method='spearman', distance='correlation',
    store_rdms=True,
)
print(result_sp_corr)

In [ ]:
# Plot Spearman ρ time courses
colors = {'stimulus_identity': '#1f77b4', 'abstract_role': '#d62728', 'conjunctive_binding': '#2ca02c'}

fig, ax = plt.subplots(figsize=(11, 4))
for i, name in enumerate(result_sp_corr.model_names):
    fits = result_sp_corr.fits[i]
    ax.plot(times, fits, label=name, color=colors.get(name), linewidth=1.8)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Spearman ρ (neural vs model RDM)')
ax.set_title('Time-resolved RSA — rule-phase epochs, correlation distance')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Time-resolved RSA — Partial regression (crossnobis)

In [ ]:
print('Running partial regression RSA (crossnobis distance)...')
result_reg_xnobis = time_resolved_rsa(
    X_rule, cids_dense, model_rdms_all, times=times,
    method='regression', distance='crossnobis',
    cv=cv, store_rdms=False,
)
print(result_reg_xnobis)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for i, name in enumerate(result_reg_xnobis.model_names):
    fits = result_reg_xnobis.fits[i]
    ax.plot(times, fits, label=name, color=colors.get(name), linewidth=1.8)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Regression β (partial, ranked neural RDM)')
ax.set_title('Time-resolved RSA — crossnobis distance, partial regression')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Neural RDM inspection

Plot neural RDMs at a few timepoints to visually verify structure.

In [ ]:
neural_rdms = result_sp_corr.neural_rdms   # (n_times, K, K)

# Pick timepoints: pre-stim, early, peak (near max Spearman ρ), late
peak_t = result_sp_corr.fits[0].argmax()  # stimulus_identity peak
t_indices = [
    np.argmin(np.abs(times - (-0.05))),
    np.argmin(np.abs(times - 0.05)),
    np.argmin(np.abs(times - 0.1)),
    peak_t,
    np.argmin(np.abs(times - 0.3)),
]
t_labels = [f'{times[i]:.3f}s' for i in t_indices]

fig, axes = plt.subplots(1, len(t_indices), figsize=(3 * len(t_indices), 3.5))
vmax = np.percentile(np.abs(neural_rdms[:, np.triu_indices(K_all, k=1)[0],
                                         np.triu_indices(K_all, k=1)[1]]), 95)
for ax, ti, tl in zip(axes, t_indices, t_labels):
    im = ax.imshow(neural_rdms[ti], cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(tl, fontsize=9)
    ax.set_xticks(range(K_all)); ax.set_xticklabels(cond_labels_all, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(K_all)); ax.set_yticklabels(cond_labels_all, fontsize=6)
plt.suptitle('Neural RDMs (correlation distance) at selected timepoints', fontsize=10)
plt.colorbar(im, ax=axes[-1], fraction=0.046)
plt.tight_layout()
plt.show()

## 8. Phase-by-rule scheme: rule_type model

Check whether rule type (ABA vs ABB) is encoded using the phase_by_rule scheme.

In [ ]:
from toolkit import assign_conditions

cond_pbr = assign_conditions(df, scheme='phase_by_rule')
print('phase_by_rule conditions:', cond_pbr['condition_names'][:6], '...')

pbr_cids  = cond_pbr['condition_id']
pbr_phase = cond_pbr['epoch_phase']

# Restrict to rule3 only (where the rule has been fully presented)
r3_mask = pbr_phase == 'rule3'
X_r3    = X_all[r3_mask]
c_r3    = pbr_cids[r3_mask]

# Dense remap
cl_r3   = np.unique(c_r3)
rm_r3   = {old: new for new, old in enumerate(sorted(cl_r3))}
c_r3_d  = np.array([rm_r3[c] for c in c_r3])
meta_r3 = cond_pbr['condition_metadata']
meta_r3 = meta_r3[meta_r3['cond_id'].isin(cl_r3)].reset_index(drop=True)
meta_r3['cond_id'] = meta_r3['cond_id'].map(rm_r3)

print(f'rule3 data: {X_r3.shape}, conditions: {len(meta_r3)}')
print(meta_r3)

In [ ]:
rdm_ruletype = {'rule_type': build_model_rdm(meta_r3, 'rule_type')}

result_rt = time_resolved_rsa(
    X_r3, c_r3_d, rdm_ruletype, times=times,
    method='spearman', distance='crossnobis',
    cv=CVSplitter(n_splits=5, stratified=True, shuffle=True, random_state=42),
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(times, result_rt.fits[0], color='#9467bd', linewidth=1.8, label='rule_type')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Spearman ρ')
ax.set_title('Rule type encoding — rule3 epochs (crossnobis distance)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary

Key outputs from this notebook:

| Analysis | What it tells you |
|---|---|
| Spearman ρ — stimulus_identity | Does MEG encode which shape is currently shown? |
| Spearman ρ — abstract_role | Does MEG encode whether the shape is playing the A or B role, ignoring its identity? |
| Spearman ρ — conjunctive_binding | Does MEG encode the binding of specific shape to specific role? |
| Partial regression β | Which model uniquely predicts neural similarity after controlling for the others? |
| Rule-type RSA on rule3 | Is the overall rule type (ABA vs ABB) decodable from neural geometry? |

**Interpretation notes**:
- Crossnobis distances can be negative (expected under null = 0); large positive values indicate the model captures real representational structure.
- Spearman ρ with a binary model that is nearly constant (e.g., abstract_role in rule1-only epochs where all conditions are A-role) will return nan/0 — always use multi-role data for that model.
- Partial regression β values are not bounded like ρ; compare their signs and relative magnitudes, not their absolute scale.